# Load UKHRD into this Lakehouse

Builds **one table per NHS reference list** — `ukhrd_admission_method`, `ukhrd_treatment_function_code`,
and 119 more — plus `ukhrd_codes` (every version, effective-dated) and `ukhrd_lists` (each list's NHS page).

**To use it:** attach this notebook to a Lakehouse, run it, then schedule it daily. It always reads the
newest UKHRD release, so the code never needs changing. The first run takes a few minutes; later runs are quick.

**Where the data comes from.** Contains information from NHS England, licensed under the current version of
the Open Government Licence. Codes and descriptions are copied exactly as published in the NHS Data Model and
Dictionary (https://www.datadictionary.nhs.uk/). UKHRD is independent and not endorsed by NHS England; the
dictionary is the authority. Keep this credit with the data.

In [ ]:
import urllib.request

from pyspark.sql import functions as F

BASE = "https://github.com/eamazon/ukhrd/releases/latest/download/"
FILES = "/lakehouse/default/Files/"   # the Lakehouse this notebook is attached to
PREFIX = "ukhrd_"                      # table name prefix: ukhrd_admission_method, ukhrd_codes, …

for name in ("codes", "lists"):
    urllib.request.urlretrieve(BASE + name + ".parquet", FILES + PREFIX + name + ".parquet")
    print("downloaded", name + ".parquet")

In [ ]:
# The two wide tables: every version of every code, and every version of every list's details.
codes = spark.read.parquet("Files/" + PREFIX + "codes.parquet").cache()
lists = spark.read.parquet("Files/" + PREFIX + "lists.parquet")

codes.write.mode("overwrite").saveAsTable(PREFIX + "codes")
lists.write.mode("overwrite").saveAsTable(PREFIX + "lists")

print(codes.count(), "code versions,", lists.count(), "list versions")

In [ ]:
# One table per reference list, holding today's codes only: code_kind, code, description.
# List names are lower-case letters, digits and underscores only, so they are safe as table names.
names = sorted(r["list_name"] for r in codes.select("list_name").distinct().collect())

for name in names:
    (codes.filter((F.col("list_name") == name) & F.col("is_current"))
          .select("code_kind", "code", "description")
          .write.mode("overwrite").saveAsTable(PREFIX + name))

print(len(names), "reference tables written, for example:", ", ".join(PREFIX + n for n in names[:3]))

In [ ]:
# Check what landed.
display(spark.sql(f"SELECT * FROM {PREFIX}admission_method ORDER BY code_kind DESC, code"))

## Using it

**What a code means today** — join the list's own table, no filters on list names:

```sql
SELECT s.*, a.description AS admission_method_name
FROM   my_spells s
LEFT JOIN ukhrd_admission_method a
       ON a.code = s.admission_method_code
      AND a.code_kind = 'national'
```

**What it meant on the date of the record** — this is what `ukhrd_codes` is for:

```sql
SELECT s.*, c.description
FROM   my_spells s
LEFT JOIN ukhrd_codes c
       ON c.list_name = 'admission_method' AND c.code_kind = 'national'
      AND c.code = s.admission_method_code
      AND s.admission_date >= c.valid_from
      AND (c.valid_to IS NULL OR s.admission_date < c.valid_to)
```

**`code_kind`** is `national` for a real code and `default` for the publisher's "not known / not applicable"
values (usually `98`, `99`). Real NHS files carry them, so drop them knowingly or not at all.

**Where a list came from** — `ukhrd_lists` holds the exact NHS page and the CDS record types that use it.

**Scheduling:** UKHRD checks the dictionary every morning and publishes a release only when NHS England
changes something. A daily run of this notebook is plenty.